In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


In [ ]:
HERE = Path.cwd()
REPO_ROOT = HERE.parents[3]
DATA_DIR = REPO_ROOT / "data" / "european_soccer_leagues" / "pure_luck_result_based"
OUT_CSV = HERE / "first_second_win_pct_all_seeds.csv"

LEAGUES = ["bundesliga", "la_liga", "premier_league", "serie_a"]

In [10]:
def load_matches(league: str) -> pd.DataFrame:
    df = pd.read_csv(DATA_DIR / f"{league}_simulated_matches_all_seeds.csv")
    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"])
    return df.sort_values(["season", "date"]).reset_index(drop=True)


In [11]:
def team_seed_win_indicator(rows: pd.DataFrame, seed_col: str, team: str) -> np.ndarray:
    """
    Returns an array with either 1/0 wins for a given team across its matches in order.
    Outcome encoding in seed_col: 1=home win, 0=draw, -1=away win.
    A win for a team is only counted when:
      - team is home AND value==+1
      - team is away AND value==-1
    """
    vals = rows[seed_col].to_numpy()
    is_home = (rows["home_team"].to_numpy() == team)
    is_away = ~is_home
    return ((is_home & (vals == 1)) | (is_away & (vals == -1))).astype(float)

def half_win_pct(win_vec: np.ndarray) -> tuple[float, float]:
    """
    Divide a team's ordered win vector into first/second halves of the season and then return mean of each.
    """
    n = len(win_vec)
    if n == 0:
        return (np.nan, np.nan)
    cut = n // 2
    first = win_vec[:cut]
    second = win_vec[cut:]
    first_pct = float(np.nan) if first.size == 0 else float(first.mean())
    second_pct = float(np.nan) if second.size == 0 else float(second.mean())
    return (first_pct, second_pct)

In [15]:
def compute_for_league(league: str) -> pd.DataFrame:
    """
    Computes first/second-half win percentages for each team across all 10 simulation seeds, 
    for each league and each season.
    """
    df = load_matches(league)
    seed_cols = [c for c in df.columns if c.startswith("simulated_home_team_result_seed_")]
    out_rows = []

    # go over each season and team
    for season, season_df in df.groupby("season", sort=True):
        teams = pd.unique(season_df[["home_team", "away_team"]].values.ravel())
        for team in np.sort(teams):
            team_rows = season_df[(season_df["home_team"] == team) | (season_df["away_team"] == team)].copy()
            team_rows = team_rows.sort_values("date") if "date" in team_rows.columns else team_rows

            rec = {"league": league, "season": season, "team": team}

            first_vals, second_vals = [], []
            for seed_col in seed_cols:
                wins = team_seed_win_indicator(team_rows, seed_col, team)
                f_pct, s_pct = half_win_pct(wins)

                seed_num = seed_col.split('_')[-1]

                rec[f"first_half_win_pct_seed_{seed_num}"] = f_pct
                rec[f"second_half_win_pct_seed_{seed_num}"] = s_pct
                
                first_vals.append(f_pct)
                second_vals.append(s_pct)

            # average across 10 seeds
            rec["first_half_win_pct_avg"] = np.nanmean(first_vals)
            rec["second_half_win_pct_avg"] = np.nanmean(second_vals)

            out_rows.append(rec)

    return pd.DataFrame(out_rows)

In [17]:
frames = [compute_for_league(lg) for lg in LEAGUES]
combined = pd.concat(frames, ignore_index=True)

# arrange columns 
fixed = ["league", "season", "team"]

# collect all seed data
seed_nums = sorted({
    int(c.split('_')[-1])
    for c in combined.columns
    if c.startswith("first_half_win_pct_seed_")
})

# order it by seeds 
ordered_seed_cols = []
for n in seed_nums:
    ordered_seed_cols.append(f"first_half_win_pct_seed_{n}")
    ordered_seed_cols.append(f"second_half_win_pct_seed_{n}")

# combine in the given order
tail = ["first_half_win_pct_avg", "second_half_win_pct_avg"]
combined = combined[fixed + ordered_seed_cols + tail]

combined.to_csv(OUT_CSV, index=False)

print(f"✅ Combined output saved at: {OUT_CSV}")
combined.head()

✅ Combined output saved at: /Users/adhvik_rayaprolu/Desktop/uiuc/IML_Fall2025/skillvsluck/output/european_soccer_leagues/correlations/pure_luck_result_based_updated/first_second_win_pct_all_seeds.csv


,league,season,team,first_half_win_pct_seed_1,second_half_win_pct_seed_1,first_half_win_pct_seed_2,second_half_win_pct_seed_2,first_half_win_pct_seed_3,second_half_win_pct_seed_3,first_half_win_pct_seed_4,...,first_half_win_pct_seed_7,second_half_win_pct_seed_7,first_half_win_pct_seed_8,second_half_win_pct_seed_8,first_half_win_pct_seed_9,second_half_win_pct_seed_9,first_half_win_pct_seed_10,second_half_win_pct_seed_10,first_half_win_pct_avg,second_half_win_pct_avg
0,bundesliga,2004,BAY,0.647059,0.235294,0.411765,0.235294,0.470588,0.235294,0.588235,...,0.411765,0.470588,0.470588,0.470588,0.294118,0.294118,0.529412,0.352941,0.476471,0.347059
1,bundesliga,2004,BIE,0.470588,0.470588,0.352941,0.470588,0.470588,0.411765,0.588235,...,0.470588,0.235294,0.529412,0.176471,0.647059,0.294118,0.117647,0.352941,0.447059,0.347059
2,bundesliga,2004,BOC,0.235294,0.176471,0.411765,0.411765,0.529412,0.235294,0.352941,...,0.529412,0.470588,0.235294,0.294118,0.235294,0.352941,0.411765,0.294118,0.394118,0.347059
3,bundesliga,2004,DOR,0.294118,0.470588,0.352941,0.411765,0.352941,0.411765,0.294118,...,0.058824,0.529412,0.470588,0.705882,0.411765,0.235294,0.235294,0.352941,0.317647,0.417647
4,bundesliga,2004,FRE,0.352941,0.352941,0.235294,0.235294,0.411765,0.352941,0.352941,...,0.352941,0.176471,0.352941,0.235294,0.588235,0.411765,0.235294,0.117647,0.370588,0.300000
